# Binary Dataset 실습

이 노트북은 `BinaryDataset`의 이미지 정규화, 홀짝 이진화 변환, `Dataloader` 배치 생성 동작을 직접 실행하고 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**데이터셋 경로**: `/mnt/d/datasets/mnist/` (train/test `.gz` 4개 파일 필요)

**목표**
- `binarize`가 정수 레이블을 홀수=1, 짝수=0으로 변환하는 과정을 확인한다.
- `BinaryDataset`이 이미지와 target을 eager하게 변환하여 `__getitem__`에서 인덱싱만 수행함을 확인한다.
- 홀수/짝수 분포와 target 시각화를 통해 이진화 결과를 직관적으로 파악한다.
- `Dataloader`로 배치를 생성하고 학습 루프에서 사용하는 방식을 시연한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.data.mnist import load_images, load_labels
from src.data import transforms as T
from src.data.datasets import BinaryDataset
from src.data.dataloader import Dataloader

## 1. 개요

`BinaryDataset`은 MNIST 숫자가 홀수인지 짝수인지 판별하는 binary classification task용 Dataset이다.
정수 레이블을 `label % 2`로 변환하여 홀수(1, 3, 5, 7, 9)를 1.0, 짝수(0, 2, 4, 6, 8)를 0.0으로 매핑한다.
이미지 변환 방식은 `MulticlassDataset`과 동일하며(`normalize + to_flat`), target 형태만 다르다.

각 변환 함수의 역할은 다음과 같다.

| 함수 | 입력 | 출력 | 역할 |
|---|---|---|---|
| `normalize` | `(N, 28, 28) uint8` | `(N, 28, 28) float32` | `/255.0` — 픽셀 값을 `[0, 1]`로 정규화 |
| `to_flat` | `(N, 28, 28) float32` | `(N, 784) float32` | `reshape` — MLP 입력 형태로 펼침 |
| `binarize` | `(N,) uint8` | `(N, 1) float32` | `label % 2` — 홀수=1.0, 짝수=0.0 |

## 2. binarize 변환 확인

`load_labels`가 반환한 원본 `uint8` 레이블 배열에 `binarize`를 적용한다.
각 레이블이 어떤 값으로 변환되는지, 홀수와 짝수가 올바르게 구분되는지 확인한다.

In [ ]:
labels_raw = load_labels("train")    # (60000,) uint8
labels_bin = T.binarize(labels_raw)  # (60000, 1) float32

print(f"raw     : shape={labels_raw.shape}, dtype={labels_raw.dtype}")
print(f"binarize: shape={labels_bin.shape}, dtype={labels_bin.dtype}")
print()
print("레이블별 변환 결과:")
for digit in range(10):
    parity = "홀수" if digit % 2 == 1 else "짝수"
    expected = digit % 2
    print(f"  label={digit} ({parity})  →  {float(expected):.1f}")

In [ ]:
# 첫 20개 샘플로 변환 결과 시각화
n_show = 20
fig, ax = plt.subplots(figsize=(12, 3))

x_pos = range(n_show)
values = labels_bin[:n_show, 0]
raw_labels = labels_raw[:n_show]
colors = ['tomato' if v == 1.0 else 'steelblue' for v in values]

bars = ax.bar(x_pos, values, color=colors, alpha=0.8, edgecolor='white')
ax.set_title("binarize transform (first 20 samples)")
ax.set_xlabel("sample index")
ax.set_ylabel("target (1=odd, 0=even)")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{i}\n(={raw_labels[i]})" for i in range(n_show)], fontsize=8)
ax.set_yticks([0, 1])
ax.set_ylim(-0.1, 1.3)
ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='tomato', label='odd (1.0)'),
                   Patch(facecolor='steelblue', label='even (0.0)')]
ax.legend(handles=legend_elements, loc='upper right')

fig.tight_layout()
plt.show()

In [ ]:
assert labels_bin.shape == (60000, 1)
assert labels_bin.dtype == np.float32
assert set(np.unique(labels_bin)).issubset({0.0, 1.0})

# 홀수 레이블 → 1.0
odd_mask = labels_raw % 2 == 1
assert np.all(labels_bin[odd_mask] == 1.0), "odd labels not mapped to 1.0"

# 짝수 레이블 → 0.0
even_mask = labels_raw % 2 == 0
assert np.all(labels_bin[even_mask] == 0.0), "even labels not mapped to 0.0"

print("binarize validation passed")

## 3. 클래스 분포 확인

MNIST train 60,000개 샘플에서 홀수와 짝수의 분포를 확인한다.
0~9 각 digit의 등장 빈도는 거의 균일하므로 홀수와 짝수 비율도 거의 50:50에 가까워야 한다.

In [ ]:
n_odd  = int(labels_bin.sum())
n_even = len(labels_bin) - n_odd
total  = len(labels_bin)

print(f"total samples: {total}")
print(f"  odd  (1.0): {n_odd:5d}  ({n_odd/total*100:.1f}%)")
print(f"  even (0.0): {n_even:5d}  ({n_even/total*100:.1f}%)")
print()
print("digit별 등장 횟수:")
for d in range(10):
    cnt = int((labels_raw == d).sum())
    parity = "홀" if d % 2 == 1 else "짝"
    print(f"  digit {d} ({parity}): {cnt}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# digit별 분포
digit_counts = [(labels_raw == d).sum() for d in range(10)]
colors_digit = ['tomato' if d % 2 == 1 else 'steelblue' for d in range(10)]
axes[0].bar(range(10), digit_counts, color=colors_digit, alpha=0.8, edgecolor='white')
axes[0].set_title("digit distribution (red=odd, blue=even)")
axes[0].set_xlabel("digit")
axes[0].set_ylabel("count")
axes[0].set_xticks(range(10))
axes[0].grid(axis='y', alpha=0.3)

# 홀짝 파이 차트
axes[1].pie([n_odd, n_even],
            labels=[f'odd\n{n_odd} ({n_odd/total*100:.1f}%)',
                    f'even\n{n_even} ({n_even/total*100:.1f}%)'],
            colors=['tomato', 'steelblue'],
            startangle=90, autopct='%1.1f%%')
axes[1].set_title("binary class distribution")

fig.tight_layout()
plt.show()

## 4. BinaryDataset 생성 및 확인

`BinaryDataset`은 인자 없이 `split`만 지정하면 `normalize + to_flat`과 `binarize`를 기본값으로 적용한다.
`__getitem__`이 반환하는 `(image, target)` tuple에서 target은 `(1,)` shape이다.

In [ ]:
train_ds = BinaryDataset("train")
test_ds  = BinaryDataset("test")

print(f"train: len={len(train_ds)}")
print(f"  images.shape={train_ds.images.shape}, images.dtype={train_ds.images.dtype}")
print(f"  targets.shape={train_ds.targets.shape}, targets.dtype={train_ds.targets.dtype}")
print()
print(f"test: len={len(test_ds)}")
print(f"  images.shape={test_ds.images.shape}, images.dtype={test_ds.images.dtype}")
print(f"  targets.shape={test_ds.targets.shape}, targets.dtype={test_ds.targets.dtype}")

img, tgt = train_ds[0]
print(f"\ntrain_ds[0]: image.shape={img.shape}, target.shape={tgt.shape}, target={tgt}")

In [ ]:
assert len(train_ds) == 60000
assert len(test_ds) == 10000

assert train_ds.images.shape == (60000, 784)
assert train_ds.images.dtype == np.float32
assert train_ds.images.min() >= 0.0 and train_ds.images.max() <= 1.0

assert train_ds.targets.shape == (60000, 1)
assert train_ds.targets.dtype == np.float32
assert set(np.unique(train_ds.targets)).issubset({0.0, 1.0})

img, tgt = train_ds[0]
assert img.shape == (784,)
assert tgt.shape == (1,)

print("BinaryDataset validation passed")

## 5. Dataloader 배치 생성

`Dataloader`는 `BinaryDataset`을 받아 `batch_size` 단위의 이미지 배치와 target 배치를 yield한다.
배치의 y shape이 `(batch_size, 1)`임을 확인한다. 이는 sigmoid 출력과 binary cross entropy 계산 시 shape을 맞추기 위해서이다.

In [ ]:
import math

BATCH_SIZE = 256
train_loader = Dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = Dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: len={len(train_loader)} batches")
print(f"test_loader : len={len(test_loader)} batches")

x_batch, y_batch = next(iter(train_loader))
print(f"\nfirst batch: x.shape={x_batch.shape}, y.shape={y_batch.shape}")
print(f"  x.dtype={x_batch.dtype}, y.dtype={y_batch.dtype}")
print(f"  unique y values: {np.unique(y_batch)}")
print(f"  y odd  count: {int(y_batch.sum())}")
print(f"  y even count: {BATCH_SIZE - int(y_batch.sum())}")

In [ ]:
# 학습 루프 시연 — 1 epoch, 첫 3 배치만 출력
print("학습 루프 시연 (1 epoch, 첫 3 배치):")
for batch_idx, (x, y) in enumerate(train_loader):
    if batch_idx < 3:
        print(f"  batch {batch_idx:3d}: x={x.shape}, y={y.shape}  "
              f"y.mean={y.mean():.3f}")
    elif batch_idx == 3:
        print("  ...")
        break

In [ ]:
assert len(train_loader) == math.ceil(len(train_ds) / BATCH_SIZE)
assert sum(len(x) for x, _ in train_loader) == len(train_ds)

x0, y0 = next(iter(train_loader))
assert x0.shape == (BATCH_SIZE, 784)
assert y0.shape == (BATCH_SIZE, 1)
assert x0.dtype == np.float32
assert y0.dtype == np.float32
assert set(np.unique(y0)).issubset({0.0, 1.0}), "batch contains unexpected target values"

print("Dataloader validation passed")

## 6. 샘플 이미지 시각화

train Dataset에서 홀수와 짝수 샘플을 각각 5개씩 꺼내 나란히 시각화한다.
같은 이미지가 원본 레이블과 이진 target 두 가지 형태로 표현됨을 확인한다.

In [ ]:
labels_raw = load_labels("train")
images_raw = load_images("train")

odd_indices  = np.where(labels_raw % 2 == 1)[0][:5]
even_indices = np.where(labels_raw % 2 == 0)[0][:5]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for col, idx in enumerate(odd_indices):
    img, tgt = train_ds[idx]
    axes[0, col].imshow(img.reshape(28, 28), cmap='Reds', vmin=0, vmax=1)
    axes[0, col].set_title(f"digit={labels_raw[idx]}\ntarget={tgt[0]:.0f}")
    axes[0, col].axis('off')

for col, idx in enumerate(even_indices):
    img, tgt = train_ds[idx]
    axes[1, col].imshow(img.reshape(28, 28), cmap='Blues', vmin=0, vmax=1)
    axes[1, col].set_title(f"digit={labels_raw[idx]}\ntarget={tgt[0]:.0f}")
    axes[1, col].axis('off')

axes[0, 0].set_ylabel("odd (target=1)", fontsize=11)
axes[1, 0].set_ylabel("even (target=0)", fontsize=11)
fig.suptitle("BinaryDataset — odd vs even samples", y=1.02)
fig.tight_layout()
plt.show()

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 항목 | 입력 | 출력 | 비고 |
|---|---|---|---|
| `binarize` | `(N,) uint8` | `(N, 1) float32` | `label % 2` — 홀수=1.0, 짝수=0.0 |
| `BinaryDataset` | `split` | `images (N,784)`, `targets (N,1)` | eager 변환 |
| `Dataloader` | `BinaryDataset` | `(x, y)` batch tuple | y shape `(B, 1)` |

**핵심 설계 원칙**
- `binarize`는 `label % 2`만으로 홀짝을 구분하므로 digit 의미를 완전히 버리고 파리티 판별 task로 전환한다.
- target shape을 `(N, 1)`로 설정하여 sigmoid 출력 `(N, 1)`과 shape이 일치하도록 한다.
- `BinaryDataset`은 기본 transform을 내장하여 인자 없이 `split`만 지정하면 바로 사용할 수 있다.